In [ ]:
import json
import torch
import random
import copy
import gc
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm
from huggingface_hub import notebook_login
notebook_login()
# ================= Configuration =================
MODEL_ID_SET= [
    # "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    # "google/gemma-2-2b-it",
    "google/gemma-3-1b-it",
    # "google/gemma-4-e2b-it",
    # "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    # "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    # "Qwen/Qwen3.5-2B-Base"
]

TRAIN_SET_FILE = "../data/cleaned/train_only.jsonl"  # demonstration pool (1,056); baseline survey: ../data/original/train_set_unified.jsonl
TEST_SET_FILE = "../data/cleaned/test_set_150_new.jsonl"  # test set (150); baseline survey: ../data/original/test_set_150.jsonl

# ================= 1. Data loading =================
def load_and_categorize_data(filepath):
    """
    Read the JSONL demonstration pool and group the items by Dimension.
    """
    categorized_data = defaultdict(list)
    
    print(f"📦 Loading and categorizing training data from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            
            user_content = ""
            for msg in data.get('messages', []):
                if msg.get('role') == 'user':
                    user_content = msg.get('content', '')
                    break
            
            # Use the Dimension field as the grouping key
            dimension = "Unknown"
            for text_line in user_content.split('\n'):
                if text_line.startswith('Dimension:'):
                    dimension = text_line.replace('Dimension:', '').strip()
                    break
            
            # Keep only the user/assistant pair (drop the system turn so it is not repeated)
            few_shot_pair = [msg for msg in data.get('messages', []) if msg['role'] in ['user', 'assistant']]
            
            if len(few_shot_pair) == 2:
                categorized_data[dimension].append(few_shot_pair)
                
    # Print the item count per dimension as a sanity check
    print("📊 Categories Distribution:")
    for dim, items in categorized_data.items():
        print(f"  - {dim}: {len(items)} items")
        
    return categorized_data

# ================= 2. N-way K-shot sampling =================
def generate_n_way_k_shot_prompt(categorized_data, K):
    """
    Strict N-way K-shot sampling (e.g. 8 dimensions x K=5 = 40 demonstrations).
    """
    sampled_pairs = []
    
    for dimension, examples in categorized_data.items():
        # If a dimension has fewer than K items, take all of them
        actual_k = min(K, len(examples))
        
        # Sample without replacement
        sampled = random.sample(examples, actual_k)
        sampled_pairs.extend(sampled)
        
    # Shuffle (user, assistant) pairs as units so questions and answers stay aligned
    random.shuffle(sampled_pairs)
    
    # Flatten into a single list of messages
    flat_few_shot_context = []
    for pair in sampled_pairs:
        flat_few_shot_context.extend(pair)
        
    return flat_few_shot_context

# ================= 3. Inference =================
# Takes the pre-built shared_few_shot_context so every model sees the same demonstrations
def generated(MODEL_ID, shared_few_shot_context, K_SHOT=1, seed=42, do_sample=False, temperature=0.7, top_p=0.9):
    
    OUTPUT_FILE = f"baseline_{MODEL_ID.split('/')[-1].replace('.', '_').lower()}_{K_SHOT}shot_seed{seed}.jsonl"
    
    SYSTEM_PROMPT = "You are a Strudel coding assistant. Write a Strudel code snippet for the following task. Return ONLY valid Strudel code, no markdown, no explanations."
    
    print(f"\nLoading model: {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        for line in tqdm(lines, desc=f"Evaluating {K_SHOT}-Shot (Seed {seed})"):
            if not line.strip(): continue
            data = json.loads(line)
            user_task = data['messages'][1]['content']
            
            messages_context = copy.deepcopy(shared_few_shot_context)
            
            if "gemma-2" in MODEL_ID.lower():
                if len(messages_context) > 0:
                    messages_context[0]["content"] = f"{SYSTEM_PROMPT}\n\n{messages_context[0]['content']}"
                    messages_context.append({"role": "user", "content": f"Task: {user_task}"})
                    messages = messages_context
                else:
                    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nTask: {user_task}"}]
            else:
                messages = [{"role": "system", "content": SYSTEM_PROMPT}]
                messages.extend(messages_context)
                messages.append({"role": "user", "content": user_task})
            
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
            
            # Reset the seed before each item so the whole batch is reproducible
            torch.manual_seed(seed)
            
            gen_kwargs = dict(
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=do_sample,
            )
            if do_sample:
                gen_kwargs.update(temperature=temperature, top_p=top_p)
            
            with torch.inference_mode():
                generated_ids = model.generate(**model_inputs, **gen_kwargs)
            
            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            clean_code = response.replace("```javascript", "").replace("```js", "").replace("```strudel", "").replace("```", "").strip()
            
            result_data = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_task},
                    {"role": "assistant", "content": clean_code}
                ]
            }
            out_f.write(json.dumps(result_data, ensure_ascii=False) + "\n")
    
    print(f"✅ Results saved to: {OUTPUT_FILE}")
    del model; del tokenizer; gc.collect(); torch.cuda.empty_cache()


# ================= 4. Run loop (same demonstrations for every model) =================

# 1. Load the demonstration pool once
global_categorized_data = load_and_categorize_data(TRAIN_SET_FILE)
SHOT_LIST= [0,1,3,5,10]
# 2. Experiment settings

SEED_LIST = [2, 4, 6, 8, 10] # five generation seeds

# 3. Loop over shot counts and generation seeds
for CURRENT_K_SHOT in SHOT_LIST:
    USE_SAMPLING = (CURRENT_K_SHOT == 0)
    for current_seed in SEED_LIST:
        print(f"\n" + "="*60)
        print(f"🎯 STARTING NEW ITERATION: K-Shot={CURRENT_K_SHOT}, SEED={current_seed}")
        print("="*60)
        
        # Build the demonstration context once per seed, before any model is loaded
        random.seed(current_seed)
        shared_few_shot_context = generate_n_way_k_shot_prompt(global_categorized_data, K=CURRENT_K_SHOT)
        print(f"✅ Generated ONE shared context with {len(shared_few_shot_context)//2} examples.")
        print(f"🔒 This exact context will be used for ALL {len(MODEL_ID_SET)} models in this seed iteration.")
        
        for MODEL_ID in MODEL_ID_SET:
            # Pass the shared demonstration context to the model
            generated(MODEL_ID, shared_few_shot_context, K_SHOT=CURRENT_K_SHOT, seed=current_seed, do_sample=USE_SAMPLING, temperature=0.7, top_p=0.9)

print("\nAll runs finished.")